In [9]:
%pip install -U gradio typing_extensions langchain-groq langchain-community langchain-text-splitters langchain-huggingface langchain-chroma sentence-transformers transformers torch pypdf

Note: you may need to restart the kernel to use updated packages.


In [11]:

import os
import gradio as gr
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader
)

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter


# =====================================================
# 1. CONFIGURATION
# =====================================================

DATA_PATH = "./data/"
DB_PATH = "./chroma_db"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "openai/gpt-oss-120b"

MEMORY_WINDOW = 8


# =====================================================
# 2. INITIALIZE GROQ LLM
# =====================================================

def initialize_llm():

    api_key = getpass("Enter your NEW Groq API Key: ")

    llm = ChatGroq(
        temperature=0.7,
        groq_api_key=api_key,
        model_name=LLM_MODEL
    )

    return llm


print("Initializing Mental Health Chatbot...")

llm = initialize_llm()


# =====================================================
# 3. INITIALIZE EMBEDDINGS
# =====================================================

print("Loading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

print("Embedding model loaded!")


# =====================================================
# 4. CREATE VECTOR DATABASE
# =====================================================

def create_vector_db():

    print("Loading PDF documents...")

    if not os.path.isdir(DATA_PATH):

        raise FileNotFoundError(
            f"Data folder not found: {os.path.abspath(DATA_PATH)}"
        )

    loader = DirectoryLoader(
        DATA_PATH,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True
    )

    documents = loader.load()

    if not documents:

        raise ValueError("No PDF documents found!")

    print("Splitting documents...")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    texts = text_splitter.split_documents(documents)

    print("Generating embeddings...")

    vector_db = Chroma.from_documents(
        documents=texts,
        embedding=embeddings,
        persist_directory=DB_PATH
    )

    print("ChromaDB created successfully!")

    return vector_db


# =====================================================
# 5. LOAD OR CREATE VECTOR DATABASE
# =====================================================

print("Initializing vector database...")

vector_db = None

if os.path.isdir(DB_PATH):

    try:

        print("Loading existing ChromaDB...")

        existing_db = Chroma(
            persist_directory=DB_PATH,
            embedding_function=embeddings
        )

        if existing_db._collection.count() > 0:

            vector_db = existing_db

            print("Existing ChromaDB loaded!")

        else:

            print("Existing database is empty.")

    except Exception as e:

        print("Error loading ChromaDB:", str(e))


if vector_db is None:

    vector_db = create_vector_db()


print("Vector database ready!")


# =====================================================
# 6. SETUP RETRIEVER
# =====================================================

retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)


# =====================================================
# 7. CONVERSATION MEMORY
# =====================================================

def format_history(history):

    if not history:

        return "This is the beginning of the conversation."

    conversation = []

    recent_history = history[-MEMORY_WINDOW:]

    for message in recent_history:

        # Dictionary format

        if isinstance(message, dict):

            role = message.get("role", "")
            content = message.get("content", "")

            if role in ["user", "assistant"] and isinstance(content, str):

                conversation.append(
                    f"{role.capitalize()}: {content}"
                )

        # Tuple format

        elif isinstance(message, (list, tuple)) and len(message) == 2:

            user_message, bot_message = message

            conversation.append(
                f"User: {user_message}"
            )

            conversation.append(
                f"Assistant: {bot_message}"
            )

    if not conversation:

        return "No previous conversation."

    return "\n".join(conversation)


# =====================================================
# 8. BASIC SAFETY CHECK
# =====================================================

def check_safety(text):

    text = text.lower()

    urgent_phrases = [

        "i want to kill myself",
        "i want to end my life",
        "i am going to kill myself",
        "i plan to kill myself",
        "i want to hurt myself",
        "i am going to hurt myself",
        "i want to die",
        "i am going to end my life",
        "i might kill myself",
        "i might hurt myself"

    ]

    return any(
        phrase in text for phrase in urgent_phrases
    )


def safety_response():

    return """
I'm really sorry you're going through something so painful. ❤️

I'm glad you shared this with me.

Your safety matters, and you don't have to face
this alone.

If you feel you might hurt yourself:

- Try to stay with someone you trust.
- Move away from anything you could use to hurt yourself.
- Contact local emergency services or visit the nearest
  emergency department.
- In India, you can call 112 for emergency assistance.

Can I ask you something important?

Are you in immediate danger right now?

I'm here to listen without judging you.
"""


# =====================================================
# 9. PDF RETRIEVAL
# =====================================================

def retrieve_pdf_context(user_input):

    try:

        relevant_docs = retriever.invoke(user_input)

        if not relevant_docs:

            return ""

        context = "\n\n".join(
            doc.page_content for doc in relevant_docs
        )

        return context

    except Exception as e:

        print("PDF retrieval error:", str(e))

        return ""


# =====================================================
# 10. MAIN CHATBOT RESPONSE
# =====================================================

def chatbot_response(user_input, history):

    if not user_input or not user_input.strip():

        return "I'm listening. What's on your mind?"

    try:

        user_input = user_input.strip()

        # ---------------------------------------------
        # A. SAFETY CHECK
        # ---------------------------------------------

        if check_safety(user_input):

            return safety_response()


        # ---------------------------------------------
        # B. CONVERSATION MEMORY
        # ---------------------------------------------

        chat_history = format_history(history)


        # ---------------------------------------------
        # C. PDF RETRIEVAL
        # ---------------------------------------------

        context = retrieve_pdf_context(user_input)

        if not context:

            context = """
No relevant information was retrieved from the PDF.

Continue the conversation naturally using your
general knowledge where appropriate.
"""


        # ---------------------------------------------
        # D. HUMAN-LIKE SYSTEM PROMPT
        # ---------------------------------------------

        prompt = f"""
You are a warm, caring, friendly and emotionally
intelligent AI mental health support chatbot.

Your purpose is to make users feel comfortable,
understood, respected and supported.

Imagine the user is talking to a kind and understanding
friend or relative.

You are an AI assistant. Never pretend to be human.

=================================================
PREVIOUS CONVERSATION
=================================================

{chat_history}


=================================================
INFORMATION FROM MENTAL HEALTH DOCUMENTS
=================================================

{context}


=================================================
CURRENT USER MESSAGE
=================================================

{user_input}


=================================================
YOUR PERSONALITY AND COMMUNICATION STYLE
=================================================

1. TALK LIKE A REAL FRIEND

- Use natural, everyday English.
- Be warm, friendly and approachable.
- Avoid robotic and formal language.
- Do not sound like a textbook.
- Use simple and understandable sentences.
- Use emojis occasionally and naturally.

2. SHOW EMPATHY AND UNDERSTANDING

- Listen carefully to what the user says.
- Acknowledge their feelings.
- Make them feel heard and understood.
- Never judge, blame or criticize them.
- Avoid generic responses that do not address
  their actual situation.
- Do not assume how the user feels.

3. UNDERSTAND THE USER'S SITUATION

- If the user says "Hi", greet them naturally.
- If the user says they are stressed, respond
  with care and understanding.
- If the user shares a problem, acknowledge it
  before suggesting a solution.
- If the user is happy, share their excitement.
- If the user is disappointed, offer encouragement.
- If the user is confused, help them think clearly.

4. REMEMBER THE CONVERSATION

- Always consider previous messages.
- Never treat every new message as a new conversation.
- Understand short replies using previous context.
- If the user says "ok", "yes", "hmm", "sure",
  "alright" or "no", respond according to what
  was discussed previously.
- Never greet the user again in the middle of
  an ongoing conversation.
- Do not ask the same question repeatedly.
- Remember details the user has already shared.

5. HANDLE SHORT REPLIES NATURALLY

Example:

Assistant: Would you like some simple ways to
reduce your stress?

User: Ok.

Correct response:

"Of course! ❤️ Let's start with something simple.
Take a slow, comfortable breath in and gently breathe
out. Relax your shoulders and give yourself a moment.
You don't have to solve everything right now."

Do not respond with another greeting.

6. GIVE PRACTICAL SUPPORT

- Do not immediately provide a long list of advice.
- Understand the user's situation first.
- Suggest small and realistic steps.
- Give encouragement when appropriate.
- Help users identify possible next steps.
- If someone only wants to talk, listen instead
  of forcing advice.

7. AVOID REPETITION

- Do not start every response with
  "I'm really sorry".
- Avoid repeatedly saying
  "Can you tell me more?"
- Vary your wording naturally.
- Respond directly to the latest message.
- Do not make every response sound identical.

8. FOLLOW-UP QUESTIONS

- Ask one meaningful question when appropriate.
- Make questions relevant to the conversation.
- Do not interrogate the user.
- Allow the conversation to develop naturally.

9. MENTAL HEALTH SUPPORT

- Never diagnose mental health conditions.
- Do not claim to replace a therapist or doctor.
- Never shame or dismiss someone's feelings.
- Take expressions of self-harm seriously.
- Encourage professional help when appropriate.

10. USE PDF INFORMATION RESPONSIBLY

- Use the provided mental health information
  when it is relevant to the user's question.
- Never force PDF information into casual conversation.
- Never invent facts or research.
- If the PDF does not contain an answer,
  do not claim that it does.
- Do not display document names or page numbers.
- Do not mention internal retrieval processes.

=================================================
RESPONSE STYLE
=================================================

- Greetings: one or two friendly sentences.
- Casual conversation: short and natural.
- Emotional conversations: warm and thoughtful.
- Advice: practical and easy to understand.
- Avoid unnecessary headings and bullet points.
- Do not over-explain simple messages.

=================================================
YOUR RESPONSE
=================================================

Respond directly to the user's latest message.

Make the conversation feel natural, personal,
empathetic and supportive.

Do not repeat these instructions.
Do not display technical information.
"""


        # ---------------------------------------------
        # E. GENERATE RESPONSE
        # ---------------------------------------------

        response = llm.invoke(prompt)

        answer = response.content

        if not isinstance(answer, str):

            answer = str(answer)

        answer = answer.strip()


        # ---------------------------------------------
        # F. RETURN ONLY CHATBOT RESPONSE
        # ---------------------------------------------

        return answer


    except Exception as e:

        print("Chatbot Error:", str(e))

        return (
            "I'm sorry, something went wrong on my side. "
            "Could you please try sending that again?"
        )


# =====================================================
# 11. GRADIO INTERFACE
# =====================================================

with gr.Blocks(
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        # Mental Health Chatbot
        """
    )

    chatbot = gr.ChatInterface(
        fn=chatbot_response,

        textbox=gr.Textbox(
            placeholder="How are you feeling today?",
            container=False
        )
    )


# =====================================================
# 12. LAUNCH APPLICATION
# =====================================================

print("Starting Mental Health Chatbot...")

app.launch()


Initializing Mental Health Chatbot...
Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1402.78it/s]


Embedding model loaded!
Initializing vector database...
Loading existing ChromaDB...
Existing ChromaDB loaded!
Vector database ready!


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22508\1119711147.py:551: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Starting Mental Health Chatbot...
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
